# FiftyOne Demo: Image Classification with InceptionV3

This notebook demonstrates how to:

* Load the flowers classification dataset into FiftyOne
* Create splits for training, validation and testing
* Download the media to the FiftyOne Media Cache ONLY ONCE
* Export ONLY labels for the FiftyOne dataset, without exporting and duplicating
media unnecessarily
* Train an InceptionV3 model to classify over the 5 classes of flowers
* Save the model weights
* Apply the trained model on the test set of images that already exist in the 
FiftyOne Media Cache
* Write the resulting predictions to a manifest, then ingest those labels as
predictions back into the original dataset, without redundant export of media or
extra copying. 

In [1]:
import os

print(os.environ.get('FIFTYONE_API_URI'))

https://demo-api.fiftyone.ai


In [2]:
from pathlib import Path
import json
import time
from typing import List, Tuple

import fiftyone as fo
import fiftyone.utils.random as four

from PIL import Image
from tqdm import tqdm
import torch
from torch import nn, optim
from torch.utils.data import Dataset, DataLoader
from torchvision import datasets, models, transforms

# Note: Media Cache Size

Ensure that the FiftyOne Media Cache size is larger than the dataset size. Note
that the default is 32 GB. 

In [3]:
# ensure the media cache config is large enough to hold the whole dataset
fo.media_cache_config.cache_size_bytes=-1 #default is 32GB

In [4]:
DATASET_DIR='gs://voxel51-test/dwiref/flowers'
# Replace dataset directory with path to Azure where dataset is saved
# Alternatively, if the dataset is stored locally, replace with local path

dataset_name="flowers_classification_dataset"

# device="cuda" if torch.cuda.is_available() else "cpu"
device = torch.device("cuda")

In [5]:
# Ingest dataset as type ImageClassificationDirectoryTree
try:
    dataset = fo.Dataset.from_dir(
        dataset_type=fo.types.ImageClassificationDirectoryTree,
        dataset_dir=DATASET_DIR,
        name=dataset_name,
        persistent=True,
        overwrite=True,
    )
except ValueError:
    print(f'A dataset with the name {dataset_name} already exists! \n',
          f'You can load the existing dataset instead of creating a new one or,',
          f'delete the existing dataset and try again.')

# If the dataset already exists, you can load it by uncommenting this line:
# dataset = fo.load_dataset(dataset_name)

# download media to media cache
dataset.download_media()

# compute metadata for performance
dataset.compute_metadata()

# Load classes and their count
flowers_classes = dataset.default_classes
num_classes = len(flowers_classes)
print(f'There are {num_classes} in this FiftyOne dataset. The classes are:',
      f'{flowers_classes}')

 100% |███████████████| 3170/3170 [3.1s elapsed, 0s remaining, 1.0K samples/s]       
Computing metadata...
 100% |███████████████| 3170/3170 [1.9s elapsed, 0s remaining, 1.7K samples/s]         
There are 5 in this FiftyOne dataset. The classes are: ['daisy', 'dandelion', 'roses', 'sunflowers', 'tulips']


In [6]:
# Confirm media cache size, then download dataset media to local cache
print(f'FiftyOne Cloud Media Cache Size in Bytes:',
      f'{fo.media_cache_config.cache_size_bytes}')

# Note: this will only work on a cloud-backed dataset
# Skip this step if DATASET_DIR is a local path
dataset.download_media()

# Print local filepath for first sample in dataset after cache download
print(f'{dataset.first().local_path}')
# /Users/dwiref/fiftyone/__cache__/media/gcs/voxel51-test/

FiftyOne Cloud Media Cache Size in Bytes: -1
/home/mithrandir/fiftyone/__cache__/media/gcs/voxel51-test/dwiref/flowers/daisy/100080576_f52e8ee070_n.jpg


To avoid exporting the dataset separately for training, we will treat the local
media cache directory as the root dataset directory for model training.

In [7]:
# Create train/test/validation splits

four.random_split(dataset, {"train": 0.7, "test": 0.2, "val": 0.1})
print(dataset.count_sample_tags())

{'train': 2219, 'val': 317, 'test': 634}


In [8]:
# Replace with path to root directory of the dataset in local media cache

cache_dir = fo.media_cache_config.cache_dir

LOCAL_DATASET_DIR = Path(f'{cache_dir}/media/gcs/voxel51-test/dwiref/flowers')

# Parameters
BATCH_SIZE = 32
EPOCHS = 20
LR = 1e-3

In [9]:
train = dataset.match_tags('train')
val = dataset.match_tags('val')
test = dataset.match_tags('test')

In [43]:
# Export a manifest of filepaths with export_media='manifest' to avoid
# exporting media and creating unwanted redundancy

# Replace the manifest pattern path with any local path where you want to
# save the exported labels. Note, this will NOT export or copy images or media
# but only the filenames and their ground truth labels

MANIFEST_PATTERN = f'/home/mithrandir/Downloads/flowers'
train.export(
    export_dir=MANIFEST_PATTERN+'/train',
    dataset_type=fo.types.FiftyOneImageClassificationDataset,
    export_media='manifest',
    classes=dataset.default_classes,
    label_field='ground_truth',
    pretty_print=True
)

val.export(
    export_dir=MANIFEST_PATTERN+'/val',
    dataset_type=fo.types.FiftyOneImageClassificationDataset,
    export_media='manifest',
    classes=dataset.default_classes,
    label_field='ground_truth',
    pretty_print=True
)

test.export(
    export_dir=MANIFEST_PATTERN+'/test',
    dataset_type=fo.types.FiftyOneImageClassificationDataset,
    export_media='manifest',
    classes=dataset.default_classes,
    label_field='ground_truth',
    pretty_print=True
)

Directory '/home/mithrandir/Downloads/flowers/train' already exists; export will be merged with existing files
 100% |███████████████| 2219/2219 [2.0s elapsed, 0s remaining, 1.1K samples/s]       
Directory '/home/mithrandir/Downloads/flowers/val' already exists; export will be merged with existing files
 100% |█████████████████| 317/317 [770.5ms elapsed, 0s remaining, 411.4 samples/s]   
Directory '/home/mithrandir/Downloads/flowers/test' already exists; export will be merged with existing files
 100% |█████████████████| 634/634 [1.1s elapsed, 0s remaining, 562.6 samples/s]         


In [13]:
LOCAL_DATASET_DIR = Path(f'{fo.media_cache_config.cache_dir}/media/gcs/voxel51-test/dwiref/flowers')

MANIFEST_PATH_TRAIN = MANIFEST_PATTERN+'/train/labels.json'
MANIFEST_PATH_VAL = MANIFEST_PATTERN+'/val/labels.json'
MANIFEST_PATH_TEST = MANIFEST_PATTERN+'/test/labels.json'

#AL Simplify these Python syntax

# 1) --- load the manifests ---
with open(MANIFEST_PATH_TRAIN, 'r') as file:
    manifest_train = json.load(file)

with open(MANIFEST_PATH_VAL, 'r') as file:
    manifest_val = json.load(file)

with open(MANIFEST_PATH_TEST, 'r') as file:
    manifest_test = json.load(file)

idx2class: List[str] = manifest_train["classes"]
class2idx = {c: i for i, c in enumerate(idx2class)}
# {'daisy': 0, 'dandelion': 1, 'roses': 2, 'sunflowers': 3, 'tulips': 4}

def idx_to_subfolder(idx: int) -> str:
    """Map label index to the sub-folder name (“sunflowers”, …)"""
    return idx2class[idx]

# 2) --- build list of (path, idx) for training and validation ---
# train
train_sample_images: List[Tuple[Path, int]] = []
for base_name, idx in manifest_train["labels"].items():
    path = Path(LOCAL_DATASET_DIR / idx_to_subfolder(idx) / f"{base_name}.jpg")
    train_sample_images.append((path, idx))

# val
val_sample_images: List[Tuple[Path, int]] = []
for base_name, idx in manifest_val["labels"].items():
    path = Path(LOCAL_DATASET_DIR / idx_to_subfolder(idx) / f"{base_name}.jpg")
    val_sample_images.append((path, idx))

# test - create a list of image paths to predict on
test_sample_images: List[Path] = []
for base_name, idx in manifest_test["labels"].items():
    path = Path(LOCAL_DATASET_DIR / idx_to_subfolder(idx) / f"{base_name}.jpg")
    test_sample_images.append(path)

# 3) --- tiny custom Dataset around a list ---
class FlowerDataset(Dataset):
    def __init__(self, items, transform=None):
        self.items = items
        self.transform = transform
    
    def __len__(self): return len(self.items)
    
    def __getitem__(self, i):
        img_path, label = self.items[i]
        img = Image.open(img_path).convert("RGB")
        if self.transform:
            img = self.transform(img)
        return img, label

In [14]:
# Augmentations & preprocessing recommended for InceptionV3
train_tf = transforms.Compose([
    transforms.RandomResizedCrop(299),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize([0.5]*3, [0.5]*3),
])

val_tf = transforms.Compose([
    transforms.Resize(320),
    transforms.CenterCrop(299),
    transforms.ToTensor(),
    transforms.Normalize([0.5]*3, [0.5]*3),
])

In [15]:
train_ds = FlowerDataset(train_sample_images, train_tf)
val_ds = FlowerDataset(val_sample_images, val_tf)

train_loader = DataLoader(train_ds, BATCH_SIZE, shuffle=True,  num_workers=0, pin_memory=True)
val_loader   = DataLoader(val_ds,   BATCH_SIZE, shuffle=False, num_workers=0, pin_memory=True)

In [16]:
# Load InceptionV3 with default pretrained weights
model = models.inception_v3(weights=models.Inception_V3_Weights.DEFAULT)

# Replace output layer with 5-unit classifier
model.fc = nn.Linear(model.fc.in_features, len(idx2class))

# Include InceptionV3's auxiliary classifier
model.AuxLogits.fc = nn.Linear(model.AuxLogits.fc.in_features, len(idx2class))
model = model.to(device)
criterion  = nn.CrossEntropyLoss()
optimizer  = optim.AdamW(model.parameters(), lr=LR)
scheduler  = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)

Downloading: "https://download.pytorch.org/models/inception_v3_google-0cc3c7bd.pth" to /home/mithrandir/.cache/torch/hub/checkpoints/inception_v3_google-0cc3c7bd.pth
100%|██████████| 104M/104M [00:05<00:00, 20.9MB/s] 


In [17]:
def unpack_inception(out):
    """
    Returns (logits, aux_logits_or_None) for both training and eval.
    """
    # in eval mode, use only only main head
    if isinstance(out, torch.Tensor):
        return out, None
    # in training mode, you get a tuple
    else:
        return out.logits, out.aux_logits

In [18]:
def run_epoch(loader, train: bool):
    model.train() if train else model.eval()
    running_loss, correct, total = 0.0, 0, 0
    for X, y in loader:
        X, y = X.to(device), y.to(device)
        if train:
            optimizer.zero_grad()
        with torch.set_grad_enabled(train):
            # may be Tensor or tuple
            out = model(X)
            logits, aux = unpack_inception(out)
            loss = criterion(logits, y)
            # only when aux present
            if train and aux is not None:
                loss += 0.4 * criterion(aux, y)
            if train:
                loss.backward()
                optimizer.step()
        running_loss += loss.item() * X.size(0)
        preds = logits.argmax(1)
        correct += (preds == y).sum().item()
        total   += y.size(0)
    return running_loss / total, correct / total


In [19]:
for epoch in range(1, EPOCHS+1):
    t0 = time.perf_counter()
    train_loss, train_acc = run_epoch(train_loader, train=True)
    val_loss,   val_acc   = run_epoch(val_loader,   train=False)
    scheduler.step()
    dt = time.perf_counter() - t0
    print(f"[{epoch:02d}/{EPOCHS}] "
          f"train loss={train_loss:.4f} acc={train_acc:.3f} | "
          f"val loss={val_loss:.4f} acc={val_acc:.3f} | "
          f"{dt:.1f}s")

[01/20] train loss=1.1629 acc=0.704 | val loss=0.8752 acc=0.779 | 11.3s
[02/20] train loss=0.9020 acc=0.771 | val loss=0.4680 acc=0.858 | 9.3s
[03/20] train loss=0.8358 acc=0.787 | val loss=0.4556 acc=0.833 | 9.3s
[04/20] train loss=0.7912 acc=0.801 | val loss=0.3670 acc=0.883 | 9.3s
[05/20] train loss=0.7128 acc=0.809 | val loss=0.3711 acc=0.883 | 9.4s
[06/20] train loss=0.6142 acc=0.841 | val loss=0.4164 acc=0.855 | 9.3s
[07/20] train loss=0.6548 acc=0.836 | val loss=0.2476 acc=0.943 | 9.3s
[08/20] train loss=0.5444 acc=0.863 | val loss=0.2537 acc=0.905 | 9.3s
[09/20] train loss=0.4890 acc=0.877 | val loss=0.2712 acc=0.909 | 9.3s
[10/20] train loss=0.4687 acc=0.872 | val loss=0.1773 acc=0.946 | 9.3s
[11/20] train loss=0.3957 acc=0.900 | val loss=0.1561 acc=0.956 | 9.3s
[12/20] train loss=0.4216 acc=0.890 | val loss=0.1809 acc=0.937 | 9.4s
[13/20] train loss=0.2998 acc=0.923 | val loss=0.1621 acc=0.950 | 9.3s
[14/20] train loss=0.2645 acc=0.930 | val loss=0.1484 acc=0.968 | 9.3s
[15/2

In [20]:
# Save model weights
# Replace the output path for where you would like to save these weights
torch.save(
    {
        "model_state_dict": model.state_dict(), 
        "class_names": idx2class
    },
    "/home/mithrandir/Voxel51/inceptionv3_flowers.pt"
)

In [21]:
# Replace these paths accordingly
preds_manifest = "/home/mithrandir/Voxel51/predictions.json"
trained_weights = "/home/mithrandir/Voxel51/inceptionv3_flowers.pt"

In [22]:
# Load model checkpoint for trained weights
ckpt = torch.load(trained_weights, map_location=device)
pred_model = models.inception_v3()
pred_model.fc = torch.nn.Linear(
    pred_model.fc.in_features,
    len(idx2class)
)
pred_model.AuxLogits.fc = nn.Linear(pred_model.AuxLogits.fc.in_features, len(idx2class))
pred_model.load_state_dict(ckpt["model_state_dict"])
pred_model.to(device).eval()

# For embeddings, extract all but the final fully connected layer
embed_model = torch.nn.Sequential(*(list(pred_model.children())[:-1]))

<ipython-input-22-35f7849f82bc>:2: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  ckpt = torch.load(trained_weights, map_location=device)
/home/mithrandir/Voxel51/dev/51t/lib

In [23]:
# Transformation to apply before predictions
test_tf = transforms.Compose([
    transforms.Resize(320), transforms.CenterCrop(299),
    transforms.ToTensor(),  transforms.Normalize([0.5]*3, [0.5]*3),
])

In [24]:
# Run predictions on the list of images from the test split, and save
# the results to a manifest that can be ingested using the FiftyOne importer
# for dataset type FiftyOneImageClassificationDataset
labels = {}
for img_path in test_sample_images:
    img_path = Path(img_path)
    x = test_tf(Image.open(img_path).convert("RGB")).unsqueeze(0).to(device)
    with torch.no_grad():
        pred = pred_model(x)
        pred_idx = pred.argmax(1).item()
    # key = base filename (no .jpg)
    key = f"{DATASET_DIR}/{img_path.parent.name}/{img_path.name}"
    labels[key] = pred_idx

In [25]:
dataset = fo.load_dataset('flowers_classification_dataset')

In [26]:
manifest_like = {"classes": dataset.default_classes, "labels": labels}
with open(preds_manifest, "w") as f:
    json.dump(manifest_like, f, indent=4)

In [27]:
# Ingest predictions on the test set back into the original dataset
test_pred_dataset = fo.Dataset.from_dir(
    dataset_type=fo.types.FiftyOneImageClassificationDataset,
    labels_path=preds_manifest,
    name='test-flowers-preds',
    overwrite=True
)

# The above ingestion puts predictions into a 'ground_truth' field
# by default. Rename the field before merging back into the original dataset
# to avoid overwriting and polluting pre-existing ground truth, otherwise
# the resulting merge will be difficult to undo
test_pred_dataset.rename_sample_field('ground_truth','preds')

# Merge prediction results back into the original dataset
dataset.merge_samples(test_pred_dataset)

 100% |█████████████████| 634/634 [2.8s elapsed, 0s remaining, 230.5 samples/s] 


In [28]:
test_pred_dataset

Name:        test-flowers-preds
Media type:  image
Num samples: 634
Persistent:  False
Tags:        []
Sample fields:
    id:               fiftyone.core.fields.ObjectIdField
    filepath:         fiftyone.core.fields.StringField
    tags:             fiftyone.core.fields.ListField(fiftyone.core.fields.StringField)
    metadata:         fiftyone.core.fields.EmbeddedDocumentField(fiftyone.core.metadata.ImageMetadata)
    created_at:       fiftyone.core.fields.DateTimeField
    last_modified_at: fiftyone.core.fields.DateTimeField
    preds:            fiftyone.core.fields.EmbeddedDocumentField(fiftyone.core.labels.Classification)

In [53]:
# Compute embeddings for all samples in the dataset, then compute_visualization
emb_model = models.inception_v3(weights=None, aux_logits=False)  # base arch
ckpt_path = "/home/mithrandir/Voxel51/inceptionv3_flowers.pt"         # <- your .pth file
state_dict = torch.load(ckpt_path, map_location=device)
emb_model.load_state_dict(state_dict, strict=False)  # strict=False skips aux heads
emb_model.to(device).eval()
emb_extractor = torch.nn.Sequential(*(list(emb_model.children())[:-1]))

# create a manifest for the whole dataset to generate embeddings with
dataset.export(
    export_dir=MANIFEST_PATTERN+'/embed',
    dataset_type=fo.types.FiftyOneImageClassificationDataset,
    export_media='manifest',
    classes=dataset.default_classes,
    label_field='ground_truth',
    pretty_print=True
)

with open(Path(MANIFEST_PATTERN+'/embed/labels.json')) as f:
    full_manifest = json.load(f)

classes = full_manifest["classes"]
labels  = full_manifest["labels"]

img_paths = [
    LOCAL_DATASET_DIR / f"{classes[idx]}/{basename}.jpg"
    for basename, idx in labels.items()
]


# Compute embeddings on the list of images from the dataset manifest, and save
iv3_embeddings = []
with torch.no_grad():
    flag_count = 0
    for p in img_paths:
        x = test_tf(Image.open(p).convert("RGB")).unsqueeze(0).to(device)
        emb = emb_extractor(x).squeeze().cpu().numpy()   # [2048]
        iv3_embeddings.append(emb)


Directory '/home/mithrandir/Downloads/flowers/embed' already exists; export will be merged with existing files


<ipython-input-53-6975b2fbe6a4>:4: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  state_dict = torch.load(ckpt_path, map_location=device)


 100% |███████████████| 3170/3170 [2.4s elapsed, 0s remaining, 1.3K samples/s]      


In [54]:
import numpy as np
iv3_embeddings = np.array(iv3_embeddings)
print(type(iv3_embeddings))
print(iv3_embeddings.shape)

<class 'numpy.ndarray'>
(3170, 2048)


In [58]:
import fiftyone.brain as fob

umap_results = fob.compute_visualization(
    dataset,
    embeddings=iv3_embeddings,
    num_dims=2,
    method='umap',
    brain_key='inceptionv3_umap',
    verbose=True,
    seed=51
)

tsne_results = fob.compute_visualization(
    dataset,
    embeddings=iv3_embeddings,
    num_dims=2,
    method='tsne',
    brain_key='inceptionv3_tsne',
    verbose=True,
    seed=51
)

pca_results = fob.compute_visualization(
    dataset,
    embeddings=iv3_embeddings,
    num_dims=2,
    method='pca',
    brain_key='inceptionv3_pca',
    verbose=True,
    seed=51
)

Generating visualization...
UMAP(n_jobs=1, random_state=51, verbose=True)
Sat May  3 17:15:57 2025 Construct fuzzy simplicial set


/home/mithrandir/Voxel51/dev/51t/lib/python3.11/site-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


Sat May  3 17:16:02 2025 Finding Nearest Neighbors
Sat May  3 17:16:03 2025 Finished Nearest Neighbor Search
Sat May  3 17:16:03 2025 Construct embedding


Epochs completed:   0%|            0/500 [00:00]

	completed  0  /  500 epochs
	completed  50  /  500 epochs
	completed  100  /  500 epochs
	completed  150  /  500 epochs
	completed  200  /  500 epochs
	completed  250  /  500 epochs
	completed  300  /  500 epochs
	completed  350  /  500 epochs
	completed  400  /  500 epochs
	completed  450  /  500 epochs
Sat May  3 17:16:05 2025 Finished embedding
Generating visualization...


/home/mithrandir/Voxel51/dev/51t/lib/python3.11/site-packages/sklearn/manifold/_t_sne.py:1162: FutureWarning: 'n_iter' was renamed to 'max_iter' in version 1.5 and will be removed in 1.7.
  warnings.warn(


[t-SNE] Computing 91 nearest neighbors...
[t-SNE] Indexed 3170 samples in 0.000s...
[t-SNE] Computed neighbors for 3170 samples in 0.110s...
[t-SNE] Computed conditional probabilities for sample 1000 / 3170
[t-SNE] Computed conditional probabilities for sample 2000 / 3170
[t-SNE] Computed conditional probabilities for sample 3000 / 3170
[t-SNE] Computed conditional probabilities for sample 3170 / 3170
[t-SNE] Mean sigma: 13254405056.902206
[t-SNE] Computed conditional probabilities in 0.282s
[t-SNE] Iteration 50: error = 72.8550644, gradient norm = 0.0316317 (50 iterations in 0.278s)
[t-SNE] Iteration 100: error = 66.8301620, gradient norm = 0.0157880 (50 iterations in 0.176s)
[t-SNE] Iteration 150: error = 64.2323761, gradient norm = 0.0100986 (50 iterations in 0.175s)
[t-SNE] Iteration 200: error = 62.7615471, gradient norm = 0.0090519 (50 iterations in 0.174s)
[t-SNE] Iteration 250: error = 61.9757729, gradient norm = 0.0065969 (50 iterations in 0.175s)
[t-SNE] KL divergence after 2

In [57]:
print(type(results))
print(results.points.shape)

<class 'fiftyone.brain.visualization.VisualizationResults'>
(3170, 2)
